# FLUX Multi-Model Architecture Inspector

**Investigates Flux-Apex-V1.flx to verify:**

| Component | Expected | Check |
|-----------|----------|-------|
| `grid_adapters.encoder` | Trained GridToWave (contrastive) | ❓ |
| `cognitive.causal_tracker` | Links from training | ❓ |
| `cognitive.rule_inducer` | Induced rules | ❓ |
| `spatial_memory` | Exploration/Curiosity fields | ❓ |

**If missing:**
1. Load `gridtowave_contrastive.pt` and inject into model
2. Train rules/links with ARC data
3. Save updated `Flux-Apex-V1.flx`

---
## Cell 1: Environment Setup

In [ ]:
%%time
import os
import sys
from pathlib import Path

# Detect environment
IN_KAGGLE = os.path.exists('/kaggle')
IN_COLAB = 'google.colab' in sys.modules

if IN_KAGGLE:
    ROOT = Path('/kaggle/working/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
elif IN_COLAB:
    ROOT = Path('/content/FLUX')
    if not ROOT.exists():
        !git clone https://github.com/Unseengap/FLUX.git {ROOT}
    os.chdir(ROOT)
else:
    ROOT = Path('.').resolve()
    while ROOT.name and not (ROOT / 'flux_utils.py').exists():
        ROOT = ROOT.parent
    if not (ROOT / 'flux_utils.py').exists():
        ROOT = Path('/Users/admin/Desktop/flux')

print(f"Environment: {'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'Local'}")
print(f"Root: {ROOT}")
os.chdir(ROOT)

# Add paths
phase_paths = [
    ROOT,
    ROOT / 'phases/phase1',
    ROOT / 'phases/phase2',
    ROOT / 'phases/phase8',
    ROOT / 'phases/phase8_8',
    ROOT / 'phases/phase9_arc',
    ROOT / 'phases/phase_unified',
]
for p in phase_paths:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

print(f"✓ Environment configured with {len(phase_paths)} paths")

## Cell 2: Load HuggingFace Token

In [ ]:
import torch
import os

# ─────────────────────────────────────────────
# Device Setup
# ─────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Device: {device}")
if device == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# ─────────────────────────────────────────────
# Load HF_TOKEN properly for all environments
# ─────────────────────────────────────────────
from flux_utils import _resolve_hf_token

HF_TOKEN = None

# 1. Kaggle secrets (highest priority in Kaggle)
if IN_KAGGLE:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from Kaggle secrets")
    except Exception as e:
        print(f"⚠ Kaggle secrets error: {e}")

# 2. Colab secrets
elif IN_COLAB:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
        if HF_TOKEN:
            print("✓ HF_TOKEN loaded from Colab secrets")
    except Exception as e:
        print(f"⚠ Colab secrets error: {e}")

# 3. Fallback to flux_utils resolver (env var, .env file, cached login)
if not HF_TOKEN:
    HF_TOKEN = _resolve_hf_token()
    if HF_TOKEN:
        print("✓ HF_TOKEN loaded from environment/cache")

# 4. Set it in environment for libraries that need it
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    print(f"  Token: {HF_TOKEN[:8]}...{HF_TOKEN[-4:]}")
    print("  ✓ Set in os.environ['HF_TOKEN']")
else:
    print("⚠ HF_TOKEN not found!")
    print("  In Kaggle: Add-ons → Secrets → Add HF_TOKEN")
    print("  Locally: export HF_TOKEN=hf_xxx or add to .env")

## Cell 3: Download Flux-Apex-V1.flx

In [ ]:
%%time
from huggingface_hub import hf_hub_download, list_repo_files

HF_REPO = "UnseenGAP/FLUX"
CHECKPOINTS_DIR = ROOT / 'checkpoints'
CHECKPOINTS_DIR.mkdir(exist_ok=True)

# List all available checkpoints on HuggingFace
print("Available checkpoints on HuggingFace:")
print("=" * 60)
try:
    files = list_repo_files(HF_REPO, token=HF_TOKEN)
    checkpoint_files = [f for f in files if 'checkpoint' in f.lower() or f.endswith('.flx') or f.endswith('.pt')]
    for f in sorted(checkpoint_files):
        print(f"  {f}")
except Exception as e:
    print(f"⚠ Could not list files: {e}")

# Download Flux-Apex-V1.flx
model_path = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'
if not model_path.exists():
    print(f"\nDownloading Flux-Apex-V1.flx...")
    try:
        hf_hub_download(
            repo_id=HF_REPO,
            filename='checkpoints/Flux-Apex-V1.flx',
            local_dir=str(ROOT),
            token=HF_TOKEN,
        )
        print(f"✓ Downloaded Flux-Apex-V1.flx")
    except Exception as e:
        print(f"✗ Download failed: {e}")
else:
    print(f"\n✓ Local Flux-Apex-V1.flx found")

# Also check for gridtowave_contrastive.pt
gtw_path = CHECKPOINTS_DIR / 'gridtowave_contrastive.pt'
if not gtw_path.exists():
    print(f"\nLooking for gridtowave_contrastive.pt...")
    try:
        hf_hub_download(
            repo_id=HF_REPO,
            filename='checkpoints/gridtowave_contrastive.pt',
            local_dir=str(ROOT),
            token=HF_TOKEN,
        )
        print(f"✓ Downloaded gridtowave_contrastive.pt")
    except Exception as e:
        print(f"⚠ gridtowave_contrastive.pt not found on HF: {e}")
else:
    print(f"✓ Local gridtowave_contrastive.pt found")

## Cell 4: Deep Inspection of Flux-Apex-V1.flx

In [ ]:
%%time
print("Loading Flux-Apex-V1.flx using FLUXModel...")
print("=" * 60)

from flux_model import FLUXModel, COMPONENT_CATEGORIES

model_path = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'

# Use FLUXModel.load() for proper handling
flux = FLUXModel.load(model_path)

# Print summary using the built-in method
flux.summary()

# Also store raw state for detailed inspection
flux_model = flux.state  # Raw dict for backward compatibility with other cells

## Cell 5: Check grid_adapters Component

In [ ]:
print("Checking grid_adapters (GridToWave/WaveToGrid)...")
print("=" * 60)

if 'grid_adapters' in flux_model:
    ga = flux_model['grid_adapters']
    print(f"✓ grid_adapters found!")
    print(f"  Keys: {list(ga.keys())}")
    
    if 'encoder' in ga:
        enc = ga['encoder']
        print(f"\n  ENCODER (GridToWave):")
        if isinstance(enc, dict):
            print(f"    Type: state_dict ({len(enc)} layers)")
            total_params = sum(v.numel() for v in enc.values() if isinstance(v, torch.Tensor))
            print(f"    Parameters: {total_params:,}")
            print(f"    Layers:")
            for k, v in list(enc.items())[:10]:
                if isinstance(v, torch.Tensor):
                    print(f"      {k}: {tuple(v.shape)}")
            if len(enc) > 10:
                print(f"      ... and {len(enc) - 10} more layers")
        else:
            print(f"    Type: {type(enc).__name__}")
    else:
        print(f"\n  ✗ ENCODER NOT FOUND — GridToWave missing!")
    
    if 'decoder' in ga:
        dec = ga['decoder']
        print(f"\n  DECODER (WaveToGrid):")
        if isinstance(dec, dict):
            print(f"    Type: state_dict ({len(dec)} layers)")
            total_params = sum(v.numel() for v in dec.values() if isinstance(v, torch.Tensor))
            print(f"    Parameters: {total_params:,}")
    else:
        print(f"\n  ✗ DECODER NOT FOUND — WaveToGrid missing!")
else:
    print(f"✗ grid_adapters NOT FOUND in model!")
    print(f"\n  This means GridToWave will use random weights.")
    print(f"  We need to load gridtowave_contrastive.pt")

## Cell 6: Check Cognitive Components

In [ ]:
print("Checking Cognitive Components...")
print("=" * 60)

# Check for cognitive layer
cognitive_keys = ['cognitive', 'causal_tracker', 'rule_inducer', 'goal_planner']
found_cognitive = {}

for key in cognitive_keys:
    if key in flux_model:
        found_cognitive[key] = flux_model[key]
        print(f"✓ {key} found")
    else:
        print(f"✗ {key} NOT FOUND")

# Check 'components' dict
if 'components' in flux_model:
    comps = flux_model['components']
    print(f"\nComponents dict:")
    for k, v in comps.items():
        status = "✓" if v else "✗"
        print(f"  {status} {k}: {v}")

# Check 'state' dict for cognitive components
if 'state' in flux_model:
    state = flux_model['state']
    print(f"\nState dict keys:")
    for k in state.keys():
        val = state[k]
        if isinstance(val, dict):
            print(f"  📁 {k}: dict ({len(val)} items)")
        elif isinstance(val, (list, tuple)):
            print(f"  📋 {k}: list ({len(val)} items)")
        else:
            print(f"  📝 {k}: {type(val).__name__}")

# Check causal_tracker specifically
print("\n" + "=" * 60)
print("CAUSAL TRACKER ANALYSIS:")

ct = None
if 'causal_tracker' in flux_model:
    ct = flux_model['causal_tracker']
elif 'state' in flux_model and 'causal_tracker' in flux_model['state']:
    ct = flux_model['state']['causal_tracker']

if ct:
    print(f"  Found causal_tracker data")
    if isinstance(ct, dict):
        print(f"  Keys: {list(ct.keys())}")
        if 'links' in ct:
            links = ct['links']
            print(f"  Links: {len(links)} causal links stored")
            if len(links) == 0:
                print(f"  ⚠ EMPTY — No trained causal relationships!")
            elif len(links) < 10:
                print(f"  ⚠ SPARSE — Only {len(links)} links (need training)")
            else:
                print(f"  ✓ Has {len(links)} learned causal links")
        if 'total' in ct:
            print(f"  Total recorded: {ct['total']}")
else:
    print(f"  ✗ No causal_tracker data found — needs training!")

## Cell 7: Check Rule Inducer

In [ ]:
print("Checking Rule Inducer...")
print("=" * 60)

ri = None
learned_rules = None

# Check various locations where rules might be stored
if 'rule_inducer' in flux_model:
    ri = flux_model['rule_inducer']
    print(f"✓ Found rule_inducer")
elif 'state' in flux_model and 'rule_inducer' in flux_model['state']:
    ri = flux_model['state']['rule_inducer']
    print(f"✓ Found rule_inducer in state")

if 'learned_rules' in flux_model:
    learned_rules = flux_model['learned_rules']
    print(f"✓ Found learned_rules")
elif 'state' in flux_model and 'learned_rules' in flux_model['state']:
    learned_rules = flux_model['state']['learned_rules']
    print(f"✓ Found learned_rules in state")

print("\nRule Inducer Analysis:")
if ri:
    if isinstance(ri, dict):
        print(f"  Keys: {list(ri.keys())}")
        if 'rules' in ri:
            rules = ri['rules']
            print(f"  Rules: {len(rules)}")
            if len(rules) == 0:
                print(f"  ⚠ EMPTY — No induced rules!")
        if 'total_rules' in ri:
            print(f"  Total rules: {ri['total_rules']}")
else:
    print(f"  ✗ No rule_inducer data")

print("\nLearned Rules Analysis:")
if learned_rules:
    if isinstance(learned_rules, dict):
        rules = learned_rules.get('rules', [])
    elif isinstance(learned_rules, list):
        rules = learned_rules
    else:
        rules = []
    
    print(f"  Total learned rules: {len(rules)}")
    if len(rules) == 0:
        print(f"  ⚠ NO RULES — Agent has no learned patterns!")
        print(f"    This means the agent cannot predict action effects.")
        print(f"    Needs training on ARC examples.")
    else:
        print(f"  Sample rules:")
        for i, rule in enumerate(rules[:5]):
            print(f"    {i+1}. {rule}")
else:
    print(f"  ✗ No learned_rules found")

## Cell 8: Check Spatial Memory

In [ ]:
print("Checking Spatial Memory (Ice/Water)...")
print("=" * 60)

sm = None
if 'spatial_memory' in flux_model:
    sm = flux_model['spatial_memory']
    print(f"✓ Found spatial_memory")
elif 'state' in flux_model and 'spatial_memory' in flux_model['state']:
    sm = flux_model['state']['spatial_memory']
    print(f"✓ Found spatial_memory in state")

if sm:
    print(f"\nSpatial Memory Contents:")
    if isinstance(sm, dict):
        for k, v in sm.items():
            if isinstance(v, torch.Tensor):
                print(f"  {k}: tensor {tuple(v.shape)}, sum={v.sum().item():.4f}")
            elif isinstance(v, dict):
                print(f"  {k}: config dict")
                for ck, cv in v.items():
                    print(f"    {ck}: {cv}")
            else:
                print(f"  {k}: {type(v).__name__}")
        
        # Check if it has real data
        if 'exploration_mass' in sm:
            mass = sm['exploration_mass']
            if mass.sum() == 0:
                print(f"\n  ⚠ exploration_mass is EMPTY (no exploration data)")
            else:
                print(f"\n  ✓ exploration_mass has {(mass > 0).sum().item()} explored cells")
else:
    print(f"✗ No spatial_memory found")

## Cell 9: Summary — What's Missing?

In [ ]:
print("="*60)
print("FLUX MULTI-MODEL HEALTH CHECK SUMMARY")
print("="*60)

issues = []
warnings = []

# Check GridToWave
if 'grid_adapters' not in flux_model or 'encoder' not in flux_model.get('grid_adapters', {}):
    issues.append("GridToWave encoder missing — using random weights!")
elif len(flux_model['grid_adapters']['encoder']) < 5:
    warnings.append("GridToWave has very few layers")
else:
    print("✓ GridToWave encoder: PRESENT")

# Check Causal Tracker
ct = flux_model.get('causal_tracker') or flux_model.get('state', {}).get('causal_tracker')
if not ct:
    issues.append("CausalTracker missing — no causal learning!")
elif isinstance(ct, dict) and len(ct.get('links', [])) < 10:
    warnings.append(f"CausalTracker has only {len(ct.get('links', []))} links (sparse)")
else:
    print("✓ CausalTracker: PRESENT")

# Check Rules
ri = flux_model.get('rule_inducer') or flux_model.get('learned_rules') or \
     flux_model.get('state', {}).get('rule_inducer') or flux_model.get('state', {}).get('learned_rules')
if not ri:
    issues.append("RuleInducer missing — no pattern learning!")
elif isinstance(ri, dict) and len(ri.get('rules', [])) == 0:
    warnings.append("RuleInducer has 0 rules (empty)")
else:
    print("✓ RuleInducer: PRESENT")

# Check Spatial Memory
sm = flux_model.get('spatial_memory') or flux_model.get('state', {}).get('spatial_memory')
if not sm:
    warnings.append("SpatialMemory missing — no exploration tracking")
else:
    print("✓ SpatialMemory: PRESENT")

print()

if issues:
    print("❌ CRITICAL ISSUES:")
    for i, issue in enumerate(issues, 1):
        print(f"   {i}. {issue}")

if warnings:
    print("\n⚠ WARNINGS:")
    for i, warning in enumerate(warnings, 1):
        print(f"   {i}. {warning}")

if not issues and not warnings:
    print("\n🎉 Model is healthy! All components present and populated.")
else:
    print("\n" + "="*60)
    print("NEXT STEPS:")
    if 'GridToWave' in str(issues):
        print("  1. Load gridtowave_contrastive.pt and inject into model")
    if 'CausalTracker' in str(issues) or 'links' in str(warnings):
        print("  2. Train causal links with ARC examples")
    if 'RuleInducer' in str(issues) or 'rules' in str(warnings):
        print("  3. Train rule inducer to learn patterns")

---
## Cell 10: Load & Inject gridtowave_contrastive.pt (if missing)

In [ ]:
print("Attempting to load gridtowave_contrastive.pt...")
print("=" * 60)

gtw_path = CHECKPOINTS_DIR / 'gridtowave_contrastive.pt'

# Also try to download from HF if not found locally
if not gtw_path.exists():
    print("Not found locally, trying HuggingFace...")
    try:
        files = list_repo_files(HF_REPO, token=HF_TOKEN)
        gtw_files = [f for f in files if 'gridtowave' in f.lower() or 'grid_to_wave' in f.lower()]
        print(f"GridToWave files on HF: {gtw_files}")
        
        for gtw_file in gtw_files:
            try:
                hf_hub_download(
                    repo_id=HF_REPO,
                    filename=gtw_file,
                    local_dir=str(ROOT),
                    token=HF_TOKEN,
                )
                print(f"✓ Downloaded {gtw_file}")
                gtw_path = ROOT / gtw_file
                break
            except:
                continue
    except Exception as e:
        print(f"⚠ HF search failed: {e}")

if gtw_path.exists():
    print(f"\n✓ Found: {gtw_path}")
    gtw_checkpoint = torch.load(str(gtw_path), map_location='cpu', weights_only=False)
    
    if isinstance(gtw_checkpoint, dict):
        print(f"  Keys: {list(gtw_checkpoint.keys())}")
        
        # The checkpoint has encoder_state_dict INSIDE it, not at top level
        if 'encoder_state_dict' in gtw_checkpoint:
            gtw_state = gtw_checkpoint['encoder_state_dict']
            print(f"  ✓ Found encoder_state_dict inside checkpoint")
            print(f"    Layers: {len(gtw_state)}")
            total_params = sum(v.numel() for v in gtw_state.values() if isinstance(v, torch.Tensor))
            print(f"    Parameters: {total_params:,}")
            
            # Also show training info
            if 'train_steps' in gtw_checkpoint:
                print(f"    Train steps: {gtw_checkpoint['train_steps']}")
            if 'similarity_targets' in gtw_checkpoint:
                print(f"    Similarity targets: {gtw_checkpoint['similarity_targets']}")
            
            INJECT_GTW = total_params > 0
        else:
            # Assume state_dict is at top level
            gtw_state = gtw_checkpoint
            total_params = sum(v.numel() for v in gtw_state.values() if isinstance(v, torch.Tensor))
            print(f"  (state_dict at top level, {total_params:,} params)")
            INJECT_GTW = total_params > 0
    else:
        print(f"  Type: {type(gtw_checkpoint).__name__}")
        INJECT_GTW = False
        gtw_state = None
else:
    print(f"\n✗ gridtowave_contrastive.pt not found")
    print(f"  Will train fresh GridToWave in Cell 11")
    INJECT_GTW = False
    gtw_state = None

## Cell 11: Inject GridToWave into Model

In [ ]:
print("Injecting GridToWave into Flux-Apex-V1...")
print("=" * 60)

if 'INJECT_GTW' in dir() and INJECT_GTW and gtw_state:
    # Ensure grid_adapters dict exists
    if 'grid_adapters' not in flux_model:
        flux_model['grid_adapters'] = {}
    
    # Inject encoder state_dict (the actual weights)
    flux_model['grid_adapters']['encoder'] = gtw_state
    
    total_params = sum(v.numel() for v in gtw_state.values() if isinstance(v, torch.Tensor))
    print(f"✓ Injected GridToWave encoder")
    print(f"  Layers: {len(gtw_state)}")
    print(f"  Parameters: {total_params:,}")
    
    # Mark as updated
    flux_model['modified'] = True
    flux_model['modified_components'] = flux_model.get('modified_components', []) + ['grid_adapters']
    print(f"✓ Marked model as modified")
    
    NEED_GTW_TRAINING = False
else:
    if 'grid_adapters' in flux_model and 'encoder' in flux_model['grid_adapters']:
        enc = flux_model['grid_adapters']['encoder']
        params = sum(v.numel() for v in enc.values() if isinstance(v, torch.Tensor))
        if params > 0:
            print(f"✓ GridToWave already present ({params:,} params)")
            NEED_GTW_TRAINING = False
        else:
            print(f"⚠ GridToWave has 0 parameters — needs training!")
            NEED_GTW_TRAINING = True
    else:
        print(f"⚠ No GridToWave to inject — will train fresh in Cell 11.5")
        NEED_GTW_TRAINING = True

## Cell 11.5: Train GridToWave (if needed)

Contrastive training on synthetic ARC-style grids:
- Different grids → low similarity (0.3)
- Modified grids → medium similarity (0.7)
- Transformed grids → high similarity (0.9)

In [ ]:
%%time
import random
import torch.nn.functional as F
from tqdm.auto import tqdm

# ─────────────────────────────────────────────────────────────
# Synthetic Grid Generators (from arc_training_giotto_kaggle)
# ─────────────────────────────────────────────────────────────

def generate_random_grid(size=10, num_objects=4, max_object_size=3):
    """Generate random ARC-style grid with colored objects."""
    grid = [[0] * size for _ in range(size)]
    
    for _ in range(num_objects):
        obj_type = random.choice(['point', 'line', 'rect', 'L'])
        color = random.randint(1, 9)
        r, c = random.randint(0, size-1), random.randint(0, size-1)
        
        if obj_type == 'point':
            grid[r][c] = color
        elif obj_type == 'line':
            length = random.randint(2, max_object_size)
            horizontal = random.choice([True, False])
            for i in range(length):
                if horizontal and c + i < size:
                    grid[r][c + i] = color
                elif not horizontal and r + i < size:
                    grid[r + i][c] = color
        elif obj_type == 'rect':
            h = random.randint(2, max_object_size)
            w = random.randint(2, max_object_size)
            for dr in range(h):
                for dc in range(w):
                    if r + dr < size and c + dc < size:
                        grid[r + dr][c + dc] = color
        elif obj_type == 'L':
            leg = random.randint(2, max_object_size)
            for i in range(leg):
                if r + i < size:
                    grid[r + i][c] = color
            for j in range(1, leg):
                if r + leg - 1 < size and c + j < size:
                    grid[r + leg - 1][c + j] = color
    
    return grid

def modify_grid(grid, num_changes=3):
    """Create modified version of grid."""
    new_grid = [row[:] for row in grid]
    size = len(grid)
    for _ in range(num_changes):
        r, c = random.randint(0, size-1), random.randint(0, size-1)
        new_grid[r][c] = random.randint(0, 9)
    return new_grid

def apply_transformation(grid, transform='none'):
    """Apply ARC-like transformation."""
    size = len(grid)
    if transform == 'rotate90':
        return [[grid[size-1-c][r] for c in range(size)] for r in range(size)]
    elif transform == 'flipH':
        return [row[::-1] for row in grid]
    elif transform == 'flipV':
        return grid[::-1]
    elif transform == 'recolor':
        c1, c2 = random.sample(range(1, 10), 2)
        return [[c2 if v == c1 else c1 if v == c2 else v for v in row] for row in grid]
    return grid

# ─────────────────────────────────────────────────────────────
# Training Configuration
# ─────────────────────────────────────────────────────────────

TRAIN_STEPS = 2000
BATCH_SIZE = 16
GRID_SIZE = 10
LR = 1e-4
GRAD_CLIP = 0.5

SIM_DIFFERENT = 0.3   # Different grids → low similarity
SIM_MODIFIED = 0.7    # Modified grids → medium similarity  
SIM_TRANSFORMED = 0.9 # Transformed grids → high similarity

if 'NEED_GTW_TRAINING' not in dir() or not NEED_GTW_TRAINING:
    print("✓ GridToWave training not needed — skipping")
else:
    print("Training GridToWave with contrastive learning...")
    print("=" * 60)
    print(f"  Steps: {TRAIN_STEPS}")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Learning rate: {LR}")
    
    # Initialize fresh GridToWave
    from grid_adapters import GridToWave
    
    WAVE_DIM = 432
    encoder = GridToWave(
        wave_dim=WAVE_DIM,
        num_colors=10,
        max_size=30,
        device=device,
    )
    encoder.to(device)
    encoder.train()
    
    params = sum(p.numel() for p in encoder.parameters())
    print(f"  Parameters: {params:,}")
    
    optimizer = torch.optim.Adam(encoder.parameters(), lr=LR)
    history = {'loss': []}
    
    # Training loop
    pbar = tqdm(range(TRAIN_STEPS), desc='Training GTW')
    
    for step in pbar:
        optimizer.zero_grad()
        total_loss = torch.tensor(0.0, device=device, requires_grad=True)
        valid = 0
        
        for _ in range(BATCH_SIZE):
            anchor = generate_random_grid(GRID_SIZE, random.randint(2, 6))
            different = generate_random_grid(GRID_SIZE, random.randint(2, 6))
            modified = modify_grid(anchor, num_changes=random.randint(1, 4))
            transformed = apply_transformation(anchor, random.choice(['rotate90', 'flipH', 'flipV', 'recolor']))
            
            w_anchor = encoder.encode(anchor, mode='holistic')
            w_different = encoder.encode(different, mode='holistic')
            w_modified = encoder.encode(modified, mode='holistic')
            w_transformed = encoder.encode(transformed, mode='holistic')
            
            if any(torch.isnan(w).any() for w in [w_anchor, w_different, w_modified, w_transformed]):
                continue
            
            sim_diff = F.cosine_similarity(w_anchor.unsqueeze(0), w_different.unsqueeze(0)).squeeze()
            sim_mod = F.cosine_similarity(w_anchor.unsqueeze(0), w_modified.unsqueeze(0)).squeeze()
            sim_trans = F.cosine_similarity(w_anchor.unsqueeze(0), w_transformed.unsqueeze(0)).squeeze()
            
            loss_diff = (sim_diff - SIM_DIFFERENT).pow(2)
            loss_mod = (sim_mod - SIM_MODIFIED).pow(2)
            loss_trans = (sim_trans - SIM_TRANSFORMED).pow(2)
            loss_mag = torch.relu(2.0 - w_anchor.norm())
            
            sample_loss = loss_diff + loss_mod + loss_trans + 0.01 * loss_mag
            total_loss = total_loss + sample_loss
            valid += 1
        
        if valid > 0:
            loss = total_loss / valid
            loss.backward()
            torch.nn.utils.clip_grad_norm_(encoder.parameters(), GRAD_CLIP)
            optimizer.step()
            history['loss'].append(loss.item())
            
            if (step + 1) % 100 == 0:
                avg_loss = sum(history['loss'][-100:]) / 100
                pbar.set_postfix({'loss': f'{avg_loss:.4f}'})
    
    print(f"\n✓ Training complete!")
    print(f"  Final loss: {sum(history['loss'][-10:])/10:.4f}")
    
    # Inject trained encoder into flux_model
    gtw_state = encoder.state_dict()
    if 'grid_adapters' not in flux_model:
        flux_model['grid_adapters'] = {}
    flux_model['grid_adapters']['encoder'] = gtw_state
    
    total_params = sum(v.numel() for v in gtw_state.values() if isinstance(v, torch.Tensor))
    print(f"  ✓ Injected trained GridToWave ({total_params:,} params)")
    
    flux_model['modified'] = True
    flux_model['modified_components'] = flux_model.get('modified_components', []) + ['grid_adapters_trained']

## Cell 12: Train Causal Links & Rules (if empty)

In [ ]:
print("Checking if cognitive training is needed...")
print("=" * 60)

# Check causal links
ct = flux_model.get('causal_tracker') or flux_model.get('state', {}).get('causal_tracker')
num_links = len(ct.get('links', [])) if isinstance(ct, dict) else 0

# Check rules
ri = flux_model.get('learned_rules') or flux_model.get('state', {}).get('learned_rules')
if isinstance(ri, dict):
    num_rules = len(ri.get('rules', []))
elif isinstance(ri, list):
    num_rules = len(ri)
else:
    num_rules = 0

print(f"Current state:")
print(f"  Causal links: {num_links}")
print(f"  Learned rules: {num_rules}")

NEEDS_TRAINING = num_links < 100 or num_rules < 10

if NEEDS_TRAINING:
    print(f"\n⚠ Training needed!")
    print(f"  Target: 100+ causal links, 10+ rules")
    print(f"  Run Cell 13 to train on ARC examples")
else:
    print(f"\n✓ Cognitive layer has sufficient training")

## Cell 13: Train Cognitive Components with SYNTHETIC Data

Generate synthetic ARC-style input→output pairs to train:
- **CausalTracker** — learns action→effect relationships
- **RuleInducer** — abstracts patterns into reusable rules

Patterns used:
- `color_change` — object changes color
- `fill_enclosed` — fill inside borders
- `extend_line` — extend partial lines
- `mirror` — reflect objects
- `remove_noise` — clean up single cells

In [ ]:
%%time
import numpy as np
import random

if 'NEEDS_TRAINING' not in dir() or not NEEDS_TRAINING:
    print("✓ Skipping cognitive training — model already has sufficient data")
else:
    print("Training Cognitive Components on SYNTHETIC ARC Data...")
    print("=" * 60)
    
    # Initialize cognitive components
    from causal_tracker import CausalTracker
    from rule_inducer import RuleInducer
    
    causal_tracker = CausalTracker(max_history=1000, device='cpu')
    rule_inducer = RuleInducer(causal_tracker=causal_tracker)
    
    # ─────────────────────────────────────────────────────────────
    # Generate SYNTHETIC ARC-style action→effect pairs
    # ─────────────────────────────────────────────────────────────
    
    def create_synthetic_arc_pair(pattern_type: str):
        """Create input/output grid pair for a simple ARC pattern."""
        size = random.randint(5, 15)
        input_grid = np.zeros((size, size), dtype=np.int32)
        output_grid = np.zeros((size, size), dtype=np.int32)
        
        if pattern_type == 'color_change':
            # Place object, then change its color
            r, c = random.randint(1, size-2), random.randint(1, size-2)
            in_color = random.randint(1, 8)
            out_color = random.randint(1, 8)
            for dr in range(-1, 2):
                for dc in range(-1, 2):
                    input_grid[r+dr, c+dc] = in_color
                    output_grid[r+dr, c+dc] = out_color
        
        elif pattern_type == 'fill_enclosed':
            # Draw border, fill inside
            r, c = random.randint(0, size-5), random.randint(0, size-5)
            border_color = random.randint(1, 8)
            fill_color = random.randint(1, 8)
            for i in range(4):
                input_grid[r, c+i] = border_color
                input_grid[r+3, c+i] = border_color
                input_grid[r+i, c] = border_color
                input_grid[r+i, c+3] = border_color
            output_grid = input_grid.copy()
            for dr in range(1, 3):
                for dc in range(1, 3):
                    output_grid[r+dr, c+dc] = fill_color
        
        elif pattern_type == 'extend_line':
            # Draw partial line, extend it
            r = random.randint(0, size-1)
            color = random.randint(1, 8)
            start_c = random.randint(0, size//2)
            end_c = random.randint(start_c+2, size-1)
            for c in range(start_c, start_c + 3):
                input_grid[r, c] = color
            for c in range(start_c, end_c):
                output_grid[r, c] = color
        
        elif pattern_type == 'mirror':
            # Object on left, mirror to right
            mid = size // 2
            color = random.randint(1, 8)
            for _ in range(random.randint(3, 8)):
                r, c = random.randint(0, size-1), random.randint(0, mid-1)
                input_grid[r, c] = color
                output_grid[r, c] = color
                output_grid[r, size - 1 - c] = color
        
        elif pattern_type == 'remove_noise':
            # Dense pattern with noise, remove single cells
            color = random.randint(1, 8)
            for r in range(2, size-2):
                for c in range(2, size-2):
                    if random.random() < 0.4:
                        input_grid[r, c] = color
                        output_grid[r, c] = color
            # Add noise to input only
            for _ in range(random.randint(5, 10)):
                nr, nc = random.randint(0, size-1), random.randint(0, size-1)
                if input_grid[nr, nc] == 0:
                    input_grid[nr, nc] = color
        
        return input_grid, output_grid, pattern_type
    
    # Generate many synthetic examples
    PATTERNS = ['color_change', 'fill_enclosed', 'extend_line', 'mirror', 'remove_noise']
    NUM_EXAMPLES = 500
    
    print(f"\nGenerating {NUM_EXAMPLES} synthetic ARC pairs...")
    
    for i in range(NUM_EXAMPLES):
        pattern = random.choice(PATTERNS)
        try:
            input_grid, output_grid, ptype = create_synthetic_arc_pair(pattern)
            
            # Find a cell that changed
            diff = np.where(input_grid != output_grid)
            if len(diff[0]) == 0:
                continue
            
            # Pick first changed cell
            idx = random.randint(0, len(diff[0]) - 1)
            change_pos = (int(diff[0][idx]), int(diff[1][idx]))
            
            # Record causal link
            causal_tracker.record(
                position=change_pos,
                action=5,  # interact
                grid_before=input_grid,
                grid_after=output_grid,
                context={'pattern': ptype},
            )
            
        except Exception as e:
            continue
        
        if (i + 1) % 100 == 0:
            print(f"  Generated {i+1}/{NUM_EXAMPLES} pairs, {len(causal_tracker.causal_links)} links")
    
    print(f"\n✓ Recorded {len(causal_tracker.causal_links)} causal links")
    
    # ─────────────────────────────────────────────────────────────
    # Induce rules from patterns
    # ─────────────────────────────────────────────────────────────
    
    print("\nInducing rules from causal links...")
    rules = rule_inducer.analyze_causal_links(force=True)
    print(f"✓ Induced {len(rules)} rules")
    
    # Sample rules
    if rules:
        print("\nSample rules:")
        for i, rule in enumerate(rules[:5]):
            print(f"  {i+1}. {str(rule)[:80]}...")
    
    # ─────────────────────────────────────────────────────────────
    # Save to model
    # ─────────────────────────────────────────────────────────────
    
    print("\nSaving cognitive components to model...")
    
    # Save causal tracker state
    links = []
    for link in causal_tracker.causal_links[-500:]:
        links.append(link.to_dict() if hasattr(link, 'to_dict') else {
            'action': getattr(link, 'trigger_action', 5),
            'position': getattr(link, 'trigger_position', (0, 0)),
            'context': getattr(link, 'context', {}),
        })
    
    if 'state' not in flux_model:
        flux_model['state'] = {}
    
    flux_model['state']['causal_tracker'] = {
        'links': links,
        'total': len(causal_tracker.causal_links),
    }
    
    # Also update top-level causal_tracker
    if 'causal_tracker' in flux_model:
        flux_model['causal_tracker']['links'] = links
        flux_model['causal_tracker']['total'] = len(causal_tracker.causal_links)
    
    # Save rules
    flux_model['state']['learned_rules'] = {
        'rules': [str(r) for r in rules],
        'total': len(rules),
    }
    
    # Also update top-level learned_rules
    if 'learned_rules' in flux_model:
        flux_model['learned_rules']['rules'] = [str(r) for r in rules]
        flux_model['learned_rules']['total'] = len(rules)
    
    flux_model['modified'] = True
    if 'modified_components' not in flux_model:
        flux_model['modified_components'] = []
    flux_model['modified_components'] += ['causal_tracker', 'rule_inducer']
    
    print(f"✓ Saved {len(links)} links and {len(rules)} rules to model")

## Cell 14: Save Updated Flux-Apex-V1.flx

In [ ]:
%%time
print("Saving updated Flux-Apex-V1.flx using FLUXModel...")
print("=" * 60)

from datetime import datetime

if not flux_model.get('modified', False):
    print("✓ No modifications made — skipping save")
else:
    # Sync modifications back to FLUXModel object
    # Add any new components that were added to flux_model dict
    if 'grid_adapters' in flux_model:
        flux.state['grid_adapters'] = flux_model['grid_adapters']
        flux.components['grid_to_wave'] = True
    
    if 'state' in flux_model:
        if 'causal_tracker' in flux_model['state']:
            flux.state['causal_tracker'] = flux_model['state']['causal_tracker']
            flux.components['causal_tracker'] = True
        if 'learned_rules' in flux_model['state']:
            flux.state['learned_rules'] = flux_model['state']['learned_rules']
            flux.components['learned_rules'] = True
    
    # Update version
    flux.version = flux.version + '-enhanced'
    flux.metadata['last_modified'] = datetime.now().isoformat()
    flux.metadata['modified_components'] = flux_model.get('modified_components', [])
    flux._modified = True
    
    # Save using FLUXModel.save()
    output_path = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'
    flux.save(output_path, overwrite=True)
    
    print(f"\nModified components: {flux_model.get('modified_components', [])}")

## Cell 15: Upload to HuggingFace (Optional)

In [ ]:
from flux_utils import HF_REPO_ID

print("Upload to HuggingFace?")
print("=" * 60)

UPLOAD = False  # Set to True to upload

if UPLOAD and HF_TOKEN:
    from huggingface_hub import HfApi
    
    output_path = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'
    
    print(f"Uploading {output_path.name}...")
    api = HfApi(token=HF_TOKEN)
    
    api.upload_file(
        path_or_fileobj=str(output_path),
        path_in_repo=f'checkpoints/{output_path.name}',
        repo_id=HF_REPO_ID,
        commit_message=f"Enhanced Flux-Apex-V1.flx — {datetime.now().isoformat()}",
    )
    
    size_mb = output_path.stat().st_size / 1e6
    print(f"✓ Uploaded to {HF_REPO_ID}")
    print(f"  Size: {size_mb:.1f} MB")
    print(f"  https://huggingface.co/{HF_REPO_ID}")
else:
    print(f"⏭ Skipping upload")
    print(f"  UPLOAD={UPLOAD}, HF_TOKEN={'set' if HF_TOKEN else 'not set'}")
    print(f"  Set UPLOAD = True above to upload")

## Cell 16: Final Verification

In [ ]:
print("="*60)
print("FINAL VERIFICATION")
print("="*60)

# Reload the saved model to verify
output_path = CHECKPOINTS_DIR / 'Flux-Apex-V1.flx'
if output_path.exists():
    print(f"\nReloading {output_path.name} to verify...")
    flux_verified = FLUXModel.load(output_path)
    flux_verified.summary()
    
    # Detailed component check
    print("\nDetailed Component Status:")
    
    # GridToWave
    if flux_verified.has_component('grid_to_wave') or 'grid_adapters' in flux_verified.state:
        gtw = flux_verified.state.get('grid_adapters', {}).get('encoder', {})
        if isinstance(gtw, dict):
            params = sum(v.numel() for v in gtw.values() if isinstance(v, torch.Tensor))
            if params > 0:
                print(f"  ✓ GridToWave: {params:,} trained parameters")
            else:
                print(f"  ⚠ GridToWave: present but 0 parameters (needs training)")
        else:
            print(f"  ✓ GridToWave: present")
    else:
        print(f"  ✗ GridToWave: MISSING")
    
    # CausalTracker
    ct = flux_verified.state.get('causal_tracker') or flux_verified.state.get('state', {}).get('causal_tracker')
    if ct:
        links = len(ct.get('links', [])) if isinstance(ct, dict) else 0
        if links > 0:
            print(f"  ✓ CausalTracker: {links} causal links")
        else:
            print(f"  ⚠ CausalTracker: present but 0 links (needs training)")
    else:
        print(f"  ✗ CausalTracker: MISSING")
    
    # RuleInducer / Learned Rules
    lr = flux_verified.state.get('learned_rules') or flux_verified.state.get('state', {}).get('learned_rules')
    if lr:
        rules = len(lr.get('rules', [])) if isinstance(lr, dict) else 0
        if rules > 0:
            print(f"  ✓ LearnedRules: {rules} rules")
        else:
            print(f"  ⚠ LearnedRules: present but 0 rules (needs training)")
    else:
        print(f"  ✗ LearnedRules: MISSING")
    
    # SpatialMemory
    sm = flux_verified.state.get('spatial_memory')
    if sm:
        print(f"  ✓ SpatialMemory: present with exploration data")
    else:
        print(f"  ✗ SpatialMemory: MISSING")
    
    print("\n" + "="*60)
    print("VERIFICATION COMPLETE ✓")
    print("="*60)
else:
    print(f"\n⚠ No saved model found at {output_path}")